In [ ]:
import ast
import re
import numpy as np
import pandas as pd
from scipy.sparse import csc_matrix, csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

# =============================================================================
# 1. LOAD DATASETS
# =============================================================================
DATA_FIXED_PATH = "Data_Fixed.csv"
DATA_SUBJECTIVE_PATH = "Data_Subjective.csv"

# Shared primary metadata dataset
dataset = pd.read_csv(DATA_FIXED_PATH)
print(f"Fixed metadata dataset loaded: {len(dataset)} games")

# Subjective user-interaction dataset
subjective_df = pd.read_csv(DATA_SUBJECTIVE_PATH)
print(f"Subjective interaction dataset loaded: {len(subjective_df)} records")


# =============================================================================
# 2. MODEL 1: KNN RECOMMENDER
# =============================================================================
def build_knn():
    df = dataset.copy()

    title_col = "title"
    tag_col = "tags"
    genre_col = "genres"
    category_col = "categories"
    filtered_tag_col = "filtered_tags"
    rating_col = "rating"
    positive_col = "positive_ratio"
    reviews_col = "user_reviews"
    price_col = "price_final"

    # DLC / Expansion filtering
    dlc_keywords = [
        r'\bdlc\b', r'\bexpansion\b', r'\bseason pass\b', r'\badd-on\b',
        r'\baddon\b', r'\bcontent pack\b', r'\bsoundtrack\b', r'\bost\b',
        r'\bartbook\b', r'hearts of stone', r'blood and wine',
        r'shadow of erdtree', r'phantom liberty'
    ]
    dlc_regex = re.compile("|".join(dlc_keywords), re.IGNORECASE)
    df = df[~df[title_col].astype(str).str.contains(dlc_regex, na=False)].reset_index(drop=True)

    # 10,000 review threshold
    df[reviews_col] = pd.to_numeric(df[reviews_col], errors="coerce").fillna(0)
    df = df[df[reviews_col] >= 10000].reset_index(drop=True)

    def parse_attributes(value):
        if isinstance(value, (list, tuple, set)):
            items = list(value)
        elif pd.isna(value) or str(value).strip() in ("", "[]"):
            return []
        else:
            raw = str(value).strip()
            try:
                parsed = ast.literal_eval(raw)
                items = list(parsed) if isinstance(parsed, (list, tuple, set)) else re.split(r"[,;]", raw)
            except (SyntaxError, ValueError):
                items = re.split(r"[,;]", raw)

        normalized = []
        for item in items:
            token = re.sub(r"[^\w]+", "_", str(item).lower().strip()).strip("_")
            if token and token not in normalized:
                normalized.append(token)
        return normalized

    df["parsed_tags"] = df[tag_col].apply(parse_attributes)
    df["parsed_genres"] = df[genre_col].apply(parse_attributes)
    df["parsed_categories"] = df[category_col].apply(parse_attributes)
    df["parsed_filtered_tags"] = df[filtered_tag_col].apply(parse_attributes)

    # Metadata Corrections
    USE_METADATA_CORRECTIONS = True
    metadata_corrections = {
        "inscryption": ["Card Game", "Deckbuilding", "Roguelike Deckbuilder", "Card Battler", "Strategy"]
    }
    existing_vocabulary = set()
    for column in ("parsed_tags", "parsed_filtered_tags"):
        for tags in df[column]:
            existing_vocabulary.update(tags)

    if USE_METADATA_CORRECTIONS:
        for game_name, additions in metadata_corrections.items():
            matches = df.index[df[title_col].astype(str).str.strip().str.lower().eq(game_name.lower())]
            for idx in matches:
                for col in ("parsed_tags", "parsed_filtered_tags"):
                    tags = list(df.at[idx, col])
                    for tag in additions:
                        token = parse_attributes([tag])[0]
                        if token in existing_vocabulary and token not in tags:
                            tags.append(token)
                    df.at[idx, col] = tags

    generic_tags = {
        "indie", "singleplayer", "multiplayer", "2d", "3d", "casual", "action",
        "adventure", "great_soundtrack", "atmospheric", "colorful", "stylized",
        "early_access", "controller"
    }
    gameplay_tags = {
        "card_game", "card_battler", "deckbuilding", "deckbuilder",
        "roguelike_deckbuilder", "roguelike", "roguelite", "turn_based",
        "turn_based_strategy", "turn_based_combat", "strategy", "tactical",
        "dungeon_crawler", "procedural_generation", "permadeath",
        "resource_management", "puzzle", "metroidvania", "survival",
        "crafting", "base_building", "automation", "tower_defense",
        "platformer", "bullet_hell", "stealth"
    }

    def weighted_text(tags):
        tokens = []
        for tag in tags:
            rep = 1 if tag in generic_tags else (4 if tag in gameplay_tags else 2)
            tokens.extend([tag] * rep)
        return " ".join(tokens)

    def plain_text(tags):
        return " ".join(tags)

    text_specs = [
        ("parsed_tags", weighted_text),
        ("parsed_genres", plain_text),
        ("parsed_categories", plain_text),
        ("parsed_filtered_tags", weighted_text)
    ]

    text_matrices = []
    for col, transform in text_specs:
        strings = df[col].apply(transform)
        vectorizer = TfidfVectorizer(
            token_pattern=r"(?u)\b\w+\b",
            sublinear_tf=col in ("parsed_tags", "parsed_filtered_tags")
        )
        text_matrices.append(vectorizer.fit_transform(strings))

    rating_map = {
        "overwhelmingly negative": 1, "very negative": 1, "negative": 1,
        "mostly negative": 2, "mixed": 3, "mostly positive": 4,
        "positive": 4, "very positive": 5, "overwhelmingly positive": 5
    }

    s_rating = df[rating_col].fillna("").astype(str).str.strip().str.lower().map(rating_map).fillna(0).astype(float)
    s_positive = pd.to_numeric(df[positive_col], errors="coerce").fillna(0).clip(0, 100)
    s_reviews = pd.to_numeric(df[reviews_col], errors="coerce").fillna(0).clip(lower=0)
    s_price = pd.to_numeric(df[price_col], errors="coerce").fillna(0).clip(lower=0)

    def normalize(series):
        values = np.asarray(series, dtype=float).reshape(-1, 1)
        return MinMaxScaler().fit_transform(values).ravel()

    norm_rating = normalize(s_rating / 5.0)
    norm_positive = normalize(s_positive / 100.0)
    norm_reviews = normalize(np.log1p(s_reviews))
    norm_price_inv = 1.0 - normalize(s_price)

    weights = [0.0, 0.20, 0.08, 0.40, 0.08, 0.08, 0.13, 0.03]
    weighted_text_matrices = [matrix * np.sqrt(weight) for matrix, weight in zip(text_matrices, weights[:4])]
    numeric_features = np.column_stack([norm_rating, norm_positive, norm_reviews, norm_price_inv]) * np.sqrt(np.array(weights[4:]))

    game_features = hstack(weighted_text_matrices + [csr_matrix(numeric_features)], format="csr")

    knn_model = NearestNeighbors(metric="euclidean", algorithm="brute")
    knn_model.fit(game_features)

    def get_base_title(title):
        return re.split(r"[:\-—]", str(title).lower().strip())[0].strip()

    def find_game(title):
        titles = df[title_col].astype(str).str.lower().str.strip()
        query = str(title).lower().strip()
        exact = df.index[titles.eq(query)]
        if len(exact):
            return int(exact[0])
        partial = df.index[titles.str.contains(query, regex=False, na=False)]
        if len(partial):
            return int(partial[0])
        return None

    def recommend_games_knn(target_titles, dataframe, top_n=10):
        target_indices = []
        for title in target_titles:
            idx = find_game(title)
            if idx is None:
                return None, f"Game '{title}' not found."
            target_indices.append(idx)

        target_rows = [dataframe.iloc[idx] for idx in target_indices]
        centroid = np.asarray(game_features[target_indices].mean(axis=0)).ravel()
        distances, indices = knn_model.kneighbors(centroid.reshape(1, -1), n_neighbors=len(dataframe))

        seen = {get_base_title(row[title_col]) for row in target_rows}
        results = []

        for distance, idx in zip(distances[0], indices[0]):
            if idx in target_indices:
                continue
            title = dataframe.iloc[idx][title_col]
            base = get_base_title(title)
            if base in seen:
                continue
            seen.add(base)

            knn_score = 1.0 / (1.0 + float(distance))
            results.append({
                "title": title,
                "distance": float(distance),
                "knn_score": knn_score,
                "rating": dataframe.iloc[idx][rating_col],
                "positive_ratio": float(s_positive.iloc[idx]),
                "user_reviews": int(s_reviews.iloc[idx]),
                "price": float(s_price.iloc[idx])
            })
            if len(results) == top_n:
                break

        return target_rows, results

    return recommend_games_knn, df

knn_recommend, knn_df = build_knn()


# =============================================================================
# 3. MODEL 2: COSINE SIMILARITY RECOMMENDER
# =============================================================================
def build_similarity():
    df = dataset.copy()

    title_col = "title"
    category_col = "categories"
    genre_col = "genres"
    tag_col = "tags"
    filtered_tag_col = "filtered_tags"
    rating_col = "rating"
    positive_ratio_col = "positive_ratio"
    user_reviews_col = "user_reviews"
    price_col = "price_final"

    dlc_keywords = [
        r'\bdlc\b', r'\bexpansion\b', r'\bseason pass\b', r'\badd-on\b',
        r'\baddon\b', r'\bcontent pack\b', r'\bsoundtrack\b', r'\bost\b',
        r'\bartbook\b', r'hearts of stone', r'blood and wine',
        r'shadow of erdtree', r'phantom liberty'
    ]
    dlc_regex = re.compile("|".join(dlc_keywords), re.IGNORECASE)
    df = df[~df[title_col].astype(str).str.contains(dlc_regex, na=False)].reset_index(drop=True)

    rating_map = {
        "overwhelmingly negative": 1, "very negative": 1, "negative": 1,
        "mostly negative": 2, "mixed": 3, "mostly positive": 4,
        "positive": 4, "very positive": 5, "overwhelmingly positive": 5
    }
    df["rating_score"] = df[rating_col].apply(lambda r: float(rating_map.get(str(r).strip().lower(), 0.0)) if pd.notna(r) else 0.0)

    df[user_reviews_col] = pd.to_numeric(df[user_reviews_col], errors="coerce").fillna(0)
    df = df[df[user_reviews_col] >= 10000].reset_index(drop=True)

    def parse_attributes(data):
        if data is None or pd.isna(data):
            return []
        if isinstance(data, (list, tuple, set)):
            values = list(data)
        else:
            raw = str(data).strip()
            if raw.lower() in ("", "[]", "nan", "none"):
                return []
            try:
                parsed = ast.literal_eval(raw)
                values = list(parsed) if isinstance(parsed, (list, tuple, set)) else re.split(r"[,;]", raw)
            except (ValueError, SyntaxError):
                values = re.split(r"[,;]", raw)

        cleaned = []
        for value in values:
            v = re.sub(r"[^\w]+", "_", str(value).strip().lower()).strip("_")
            if v:
                cleaned.append(v)
        return list(dict.fromkeys(cleaned))

    df["parsed_tags"] = df[tag_col].apply(parse_attributes)
    df["parsed_genres"] = df[genre_col].apply(parse_attributes)
    df["parsed_categories"] = df[category_col].apply(parse_attributes)
    df["parsed_filtered_tags"] = df[filtered_tag_col].apply(parse_attributes)

    generic_tags = {
        "indie", "singleplayer", "multiplayer", "2d", "3d", "casual", "action",
        "adventure", "great_soundtrack", "atmospheric", "colorful", "stylized",
        "early_access", "controller", "steam_achievements", "steam_cloud",
        "partial_controller_support", "full_controller_support"
    }
    gameplay_tags = {
        "card_game", "card_battler", "deckbuilding", "deckbuilder",
        "roguelike_deckbuilder", "roguelike", "roguelite", "turn_based",
        "turn_based_strategy", "turn_based_combat", "strategy", "tactical",
        "tactical_rpg", "dungeon_crawler", "procedural_generation", "perma_death",
        "permadeath", "resource_management", "puzzle", "puzzle_platformer",
        "metroidvania", "souls_like", "survival", "crafting", "base_building",
        "colony_sim", "city_builder", "automation", "management", "farming_sim",
        "life_sim", "tower_defense", "real_time_strategy", "grand_strategy",
        "simulation", "rhythm", "platformer", "precision_platformer",
        "bullet_hell", "hack_and_slash", "loot", "stealth", "immersive_sim",
        "first_person_shooter", "third_person_shooter", "fighting", "racing",
        "sports", "visual_novel"
    }

    def weighted_tag_text(attributes):
        tokens = []
        for tag in attributes:
            rep = 1 if tag in generic_tags else (4 if tag in gameplay_tags else 2)
            tokens.extend([tag] * rep)
        return " ".join(tokens)

    df["clean_tag_string"] = df["parsed_tags"].apply(weighted_tag_text)
    df["clean_genre_string"] = df["parsed_genres"].apply(lambda x: " ".join(x))
    df["clean_category_string"] = df["parsed_categories"].apply(lambda x: " ".join(x))
    df["clean_filtered_tag_string"] = df["parsed_filtered_tags"].apply(weighted_tag_text)

    tfidf_tags_matrix = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b", sublinear_tf=True).fit_transform(df["clean_tag_string"])
    tfidf_genre_matrix = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b").fit_transform(df["clean_genre_string"])
    tfidf_category_matrix = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b").fit_transform(df["clean_category_string"])
    tfidf_filtered_tags_matrix = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b", sublinear_tf=True).fit_transform(df["clean_filtered_tag_string"])

    s_rating = df["rating_score"].copy()
    s_positive_ratio = pd.to_numeric(df[positive_ratio_col], errors="coerce").fillna(0.0).clip(lower=0, upper=100)
    s_user_reviews = pd.to_numeric(df[user_reviews_col], errors="coerce").fillna(0.0).clip(lower=0)
    s_price = pd.to_numeric(df[price_col], errors="coerce").fillna(0.0).clip(lower=0)

    norm_rating = MinMaxScaler().fit_transform((s_rating / 5.0).to_numpy().reshape(-1, 1)).flatten()
    norm_positive_ratio = MinMaxScaler().fit_transform((s_positive_ratio / 100.0).to_numpy().reshape(-1, 1)).flatten()
    norm_user_reviews = MinMaxScaler().fit_transform(np.log1p(s_user_reviews.to_numpy()).reshape(-1, 1)).flatten()
    norm_price_inv = 1.0 - MinMaxScaler().fit_transform(s_price.to_numpy().reshape(-1, 1)).flatten()

    def get_base_title(title):
        return re.split(r"[:\-—]", str(title).lower().strip())[0].strip()

    def recommend_games(target_titles, dataframe, top_n=10):
        target_indices = []
        target_rows = []

        for target_title in target_titles:
            query = target_title.lower().strip()
            exact_mask = dataframe[title_col].astype(str).str.lower().str.strip().eq(query)
            matched_indices = dataframe[exact_mask].index

            if len(matched_indices) == 0:
                partial_mask = dataframe[title_col].astype(str).str.lower().str.contains(query, regex=False, na=False)
                matched_indices = dataframe[partial_mask].index

            if len(matched_indices) == 0:
                return None, f"Game containing '{target_title}' was not found in the dataset."

            target_idx = matched_indices[0]
            target_indices.append(target_idx)
            target_rows.append(dataframe.iloc[target_idx])

        tag_vectors = cosine_similarity(tfidf_tags_matrix[target_indices], tfidf_tags_matrix)
        genre_vectors = cosine_similarity(tfidf_genre_matrix[target_indices], tfidf_genre_matrix)
        category_vectors = cosine_similarity(tfidf_category_matrix[target_indices], tfidf_category_matrix)
        filtered_tag_vectors = cosine_similarity(tfidf_filtered_tags_matrix[target_indices], tfidf_filtered_tags_matrix)

        combined_tag_similarity = np.mean(tag_vectors, axis=0)
        combined_genre_similarity = np.mean(genre_vectors, axis=0)
        combined_category_similarity = np.mean(category_vectors, axis=0)
        combined_filtered_tag_similarity = np.mean(filtered_tag_vectors, axis=0)

        w_tags, w_genre, w_category, w_filtered_tags = 0.00, 0.25, 0.05, 0.40
        w_rating, w_positive_ratio, w_user_reviews, w_price = 0.08, 0.08, 0.13, 0.01

        hybrid_scores = (
            (w_tags * combined_tag_similarity) +
            (w_genre * combined_genre_similarity) +
            (w_category * combined_category_similarity) +
            (w_filtered_tags * combined_filtered_tag_similarity) +
            (w_rating * norm_rating) +
            (w_positive_ratio * norm_positive_ratio) +
            (w_user_reviews * norm_user_reviews) +
            (w_price * norm_price_inv)
        )

        for idx in target_indices:
            hybrid_scores[idx] = -1.0

        sorted_indices = np.argsort(hybrid_scores)[::-1]
        seen_base_titles = {get_base_title(row[title_col]) for row in target_rows}
        results = []

        for idx in sorted_indices:
            title = dataframe.iloc[idx][title_col]
            base_title = get_base_title(title)
            if base_title in seen_base_titles:
                continue
            seen_base_titles.add(base_title)

            results.append({
                "title": title,
                "composite_score": round(float(hybrid_scores[idx]), 4),
                "tag_sim": round(float(combined_tag_similarity[idx]), 4),
                "genre_sim": round(float(combined_genre_similarity[idx]), 4),
                "category_sim": round(float(combined_category_similarity[idx]), 4),
                "filtered_tag_sim": round(float(combined_filtered_tag_similarity[idx]), 4),
                "rating": dataframe.iloc[idx][rating_col],
                "positive_ratio": float(s_positive_ratio.iloc[idx]),
                "user_reviews": int(s_user_reviews.iloc[idx]),
                "price": float(s_price.iloc[idx])
            })
            if len(results) == top_n:
                break

        return target_rows, results

    return recommend_games, df

sim_recommend, sim_df = build_similarity()


# =============================================================================
# 4. MODEL 3: IMPLICIT ALS RECOMMENDER (DIRECT DATASET TRAIN, NO PICKLE/CACHE)
# =============================================================================
def build_implicit_als():
    recomm_data = subjective_df.copy()

    # Link app_id to titles using Data_Fixed.csv
    app_id_col = "app_id" if "app_id" in dataset.columns else dataset.columns[0]
    game_title = dict(zip(dataset[app_id_col], dataset["title"]))
    title_to_app_id_map = {
        str(title).strip().lower(): app_id
        for app_id, title in game_title.items() if pd.notna(title)
    }

    recomm_data['recommendations'] = np.log1p(
        pd.to_numeric(recomm_data['hours'], errors='coerce').fillna(0).clip(lower=0)
    ) + (pd.to_numeric(recomm_data['is_recommended'], errors='coerce').fillna(0).astype(float) * 1.0)

    recomm_data = recomm_data[recomm_data['recommendations'] > 0].copy()

    user_ids = recomm_data['user_id'].astype("category")
    app_ids = recomm_data['app_id'].astype("category")

    user_index_map = dict(enumerate(user_ids.cat.categories))
    app_index_map = dict(enumerate(app_ids.cat.categories))
    app_id_to_index_map = {v: k for k, v in app_index_map.items()}

    recomm_data['user_index'] = user_ids.cat.codes
    recomm_data['app_index'] = app_ids.cat.codes

    user_app_matrix = csr_matrix(
        (recomm_data['recommendations'].astype(np.float32), 
         (recomm_data['user_index'], recomm_data['app_index'])),
        shape=(len(user_ids.cat.categories), len(app_ids.cat.categories))
    )
    user_app_matrix.sum_duplicates()

    # Self-contained ALS Matrix Factorization solver for browser & local Python
    class PureALS:
        def __init__(self, factors=32, iterations=10, regularization=0.05, alpha=40.0):
            self.factors = factors
            self.iterations = iterations
            self.regularization = regularization
            self.alpha = alpha

        def fit(self, user_items):
            n_users, n_items = user_items.shape
            rng = np.random.default_rng(42)
            self.user_factors = rng.normal(0, 0.01, (n_users, self.factors)).astype(np.float32)
            self.item_factors = rng.normal(0, 0.01, (n_items, self.factors)).astype(np.float32)
            reg = self.regularization * np.eye(self.factors, dtype=np.float32)

            for _ in range(self.iterations):
                item_gram = self.item_factors.T @ self.item_factors
                for u in range(n_users):
                    row = user_items.getrow(u)
                    if row.nnz == 0:
                        continue
                    ids = row.indices
                    factors = self.item_factors[ids]
                    c = self.alpha * row.data
                    A = item_gram + (factors.T * c) @ factors + reg
                    b = factors.T @ (1.0 + c)
                    self.user_factors[u] = np.linalg.solve(A, b)

                user_gram = self.user_factors.T @ self.user_factors
                item_csc = user_items.tocsc()
                for i in range(n_items):
                    col = item_csc.getcol(i)
                    if col.nnz == 0:
                        continue
                    ids = col.indices
                    factors = self.user_factors[ids]
                    c = self.alpha * col.data
                    A = user_gram + (factors.T * c) @ factors + reg
                    b = factors.T @ (1.0 + c)
                    self.item_factors[i] = np.linalg.solve(A, b)
            return self

        def recommend(self, user_items, N=10):
            row = user_items.tocsr()
            liked = row.indices
            if len(liked) == 0:
                return np.array([]), np.array([])
            selected = self.item_factors[liked]
            conf = 1.0 + self.alpha * row.data
            A = (selected.T * conf) @ selected + self.regularization * np.eye(self.factors, dtype=np.float32)
            b = selected.T @ conf
            u_vec = np.linalg.solve(A, b)
            scores = self.item_factors @ u_vec
            scores[liked] = -np.inf
            count = min(int(N), len(scores))
            best = np.argpartition(scores, -count)[-count:]
            best = best[np.argsort(scores[best])[::-1]]
            return best, scores[best]

    model = PureALS(factors=32, iterations=10, regularization=0.05).fit(user_app_matrix)

    def recommend_implicit(game_names, top_n=100):
        valid_app_ids = []
        for name in game_names:
            search_name = str(name).strip().lower()
            app_id = title_to_app_id_map.get(search_name)

            if app_id is None or app_id not in app_id_to_index_map:
                matches = [t for t in title_to_app_id_map if search_name in t]
                for match in matches:
                    candidate_id = title_to_app_id_map[match]
                    if candidate_id in app_id_to_index_map:
                        app_id = candidate_id
                        break

            if app_id is not None and app_id in app_id_to_index_map:
                if app_id not in valid_app_ids:
                    valid_app_ids.append(app_id)

        if not valid_app_ids:
            return None, "No input games matched in the subjective interaction dataset."

        id_indices = [app_id_to_index_map[aid] for aid in valid_app_ids]
        user_items = csr_matrix(
            (np.ones(len(id_indices), dtype=np.float32), ([0] * len(id_indices), id_indices)),
            shape=(1, len(app_index_map))
        )

        ids, scores = model.recommend(user_items=user_items, N=top_n)

        results = []
        seen_titles = set()
        for idx, score in zip(ids, scores):
            rec_app_id = app_index_map[idx]
            rec_title = game_title.get(rec_app_id, f"App ID: {rec_app_id}")
            key = str(rec_title).strip().casefold()
            if key not in seen_titles:
                seen_titles.add(key)
                results.append({"title": rec_title, "implicit_score": float(score)})

        return None, results

    return recommend_implicit

implicit_recommend = build_implicit_als()


# =============================================================================
# 5. THREE-MODEL HYBRID: 3x3 JACCARD MATRIX + RANK FUSION
# =============================================================================
CANDIDATES_PER_MODEL = 100
FINAL_TOP_N = 10
RRF_K = 60

def game_key(title):
    return str(title).strip().casefold()

def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if a | b else 0.0

def combine_recommendations(model_results, top_n=10):
    names = list(model_results)
    rank_maps = {}
    display_titles = {}

    for name, recommendations in model_results.items():
        ranks = {}
        for position, rec in enumerate(recommendations, start=1):
            key = game_key(rec['title'])
            if key not in ranks:
                ranks[key] = position
                display_titles.setdefault(key, rec['title'])
        rank_maps[name] = ranks

    # Generate 3x3 Jaccard Overlap Matrix
    overlap = pd.DataFrame(
        [[jaccard(rank_maps[a], rank_maps[b]) for b in names] for a in names],
        index=names, columns=names
    )

    raw_weights = {
        name: 1 / (1 + sum(overlap.loc[name, other] for other in names if other != name))
        for name in names
    }
    total_weight = sum(raw_weights.values())
    weights = {name: raw_weights[name] / total_weight for name in names}

    rows = []
    for key, title in display_titles.items():
        score = sum(weights[name] / (RRF_K + rank_maps[name][key]) for name in names if key in rank_maps[name])
        row = {'Game': title, 'Fusion score': score, 'Models agreeing': sum(key in rank_maps[n] for n in names)}
        for name in names:
            row[f'{name} rank'] = rank_maps[name].get(key, None)
        rows.append(row)

    ranking = pd.DataFrame(rows).sort_values(
        ['Fusion score', 'Models agreeing', 'Game'],
        ascending=[False, False, True]
    ).head(top_n).reset_index(drop=True)
    ranking.index += 1
    ranking.index.name = 'Rank'

    return ranking, overlap, weights


# =============================================================================
# 6. EXECUTION PIPELINE
# =============================================================================
print('\nEnter one game per line. Press Enter on a blank line to finish.')
selected_games = []
while True:
    title = input(f'Game {len(selected_games) + 1}: ').strip()
    if not title:
        break
    if game_key(title) not in {game_key(g) for g in selected_games}:
        selected_games.append(title)

if not selected_games:
    print('No games entered.')
else:
    _, knn_results = knn_recommend(selected_games, knn_df, top_n=CANDIDATES_PER_MODEL)
    _, sim_results = sim_recommend(selected_games, sim_df, top_n=CANDIDATES_PER_MODEL)
    _, implicit_results = implicit_recommend(selected_games, top_n=CANDIDATES_PER_MODEL)

    if not isinstance(knn_results, list):
        print('KNN Error:', knn_results)
    elif not isinstance(sim_results, list):
        print('Similarity Error:', sim_results)
    elif not isinstance(implicit_results, list):
        print('Implicit ALS Error:', implicit_results)
    else:
        model_results = {
            'KNN': knn_results,
            'Similarity': sim_results,
            'Implicit ALS': implicit_results
        }
        ranking, overlap, weights = combine_recommendations(model_results, top_n=FINAL_TOP_N)

        print('\n3x3 JACCARD OVERLAP MATRIX')
        print(overlap.round(4).to_string())
        print('\nMODEL WEIGHTS:', {k: round(v, 4) for k, v in weights.items()})
        print('\nFINAL HYBRID TOP 10')
        print(ranking.to_string(float_format=lambda x: f'{x:.6f}'))
        print('\nNote: Jaccard measures agreement; weighted rank fusion determines order.')